In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [2]:
train_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train3')
valid_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-valid3')
test_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-test3')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

▸,:,


In [4]:
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and transform data
def apply_vector_size_hint(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 1% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")

        # Apply VectorSizeHint if the vector size is greater than 0
        if vector_size > 0:
            size_hint = VectorSizeHint(inputCol=col_name, size=vector_size)
            pipeline_size_hint = Pipeline(stages=[size_hint])
            
            print(f"Fitting VectorSizeHint for column '{col_name}' on sample of {sample_fraction * 100}% of data...")
            model_size_hint = pipeline_size_hint.fit(sampled_df)
            
            print(f"Applying VectorSizeHint for column '{col_name}' on full dataset...")
            return model_size_hint.transform(data)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")

    return data

# Applying the function to each vector column
for col_name in vector_cols:
    print(f"Applying VectorSizeHint to vector column: '{col_name}'")
    train_data = apply_vector_size_hint(train_data, col_name)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")
pipeline_final_assembler = Pipeline(stages=[final_assembler])
model_final_assembler = pipeline_final_assembler.fit(train_data)
print("Pipeline Processing Done")

# Step 6: Transform train, valid, and test datasets to include the final features
train_data = model_final_assembler.transform(train_data)

print("Pipeline Transformation Done")

▸,:,


Applying VectorSizeHint to vector column: 'race_vector'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Column 'race_vector' vector size: 1
Fitting VectorSizeHint for column 'race_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'race_vector' on full dataset...


...

Column 'gender_vector' vector size: 1
Fitting VectorSizeHint for column 'gender_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'gender_vector' on full dataset...


<IPython.core.display.Javascript object>

Final input columns for feature assembly: ['race_vector', 'gender_vector', 'age_of_TBI_diagnosis', 'MedicalHistory', 'E72', 'Q02', 'S37', 'R52', 'L74', 'T50', 'S87', 'S41', 'C25', 'V58', 'B37', 'T58', 'V41', 'X34', 'C86', 'I01', 'Q11', 'M75', 'B70', 'Q52', 'W27', 'R36', 'V65', 'V28', 'I47', 'K87', 'I52', 'E86', 'Z33', 'Q23', 'Q86', 'N46', 'L62', 'V97', 'T30', 'L50', 'S56', 'H62', 'C91', 'G70', 'Q45', 'T27', 'Z77', 'S51', 'P05', 'F29', 'A83', 'F20', 'W89', 'B68', 'O72', 'X77', 'L87', 'X39', 'B42', 'R59', 'V38', 'X04', 'J35', 'V94', 'W99', 'C54', 'N88', 'F02', 'G63', 'K06', 'J22', 'D78', 'K60', 'W18', 'D21', 'H16', 'L54', 'F11', 'D04', 'H30', 'Z20', 'V73', 'G06', 'K80', 'M19', 'T07', 'W52', 'M80', 'X82', 'H65', 'Z86', 'Y75', 'E55', 'G81', 'V63', 'E87', 'D36', 'O43', 'Z51', 'L91', 'V48', 'K22', 'E73', 'D80', 'T38', 'T84', 'H27', 'Q07', 'E11', 'B00', 'G23', 'E96', 'F33', 'N05', 'Z78', 'D06', 'D74', 'I51', 'B18', 'B58', 'I69', 'R87', 'W50', 'R53', 'C4A', 'O91', 'P02', 'P37', 'M22', 'S81', '

<IPython.core.display.Javascript object>

Pipeline Transformation Done


In [5]:
valid_data = model_final_assembler.transform(valid_data)
test_data = model_final_assembler.transform(test_data)

▸,:,


In [6]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))

▸,:,


In [7]:
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

▸,:,


In [ ]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

In [ ]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
# count_zeros = 705565
# count_ones = 106624
count_zeros = 706360
count_ones = 106714

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

In [8]:
train_data_balanced = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train2-balanced')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.functions import udf
from pyspark.ml.classification import GBTClassifier
from pyspark.sql.types import DoubleType

# Train GBTClassifier
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxDepth=10, maxIter=10)
model = gbt.fit(train_data_balanced)

# Define a UDF to extract the probability of class 1
def extract_prob(prob_vector):
    if isinstance(prob_vector, float):  # If it's already a float, return as-is
        return prob_vector
    if prob_vector is not None:
        return float(prob_vector[1])  # Extract the positive class probability
    return 0.0  # Default if None

extract_prob_udf = udf(extract_prob, DoubleType())

def calculate_confusion_matrix_metrics(predictions: DataFrame, label_col: str, pred_col: str):
    """Compute confusion matrix metrics including precision, recall, specificity, F1-score, and accuracy."""
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(pred_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(pred_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(pred_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(pred_col) == 0)).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Sensitivity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1_score": f1_score,
        "J_statistic": recall + specificity - 1,  # Youden's J statistic
    }

def evaluate_threshold(predictions: DataFrame, label_col: str, probability_col: str, threshold: float):
    """Evaluate model performance at a given threshold."""
    predictions = predictions.withColumn("prob_positive", extract_prob_udf(F.col(probability_col)))
    predictions = predictions.withColumn("pred_label", (F.col("prob_positive") >= threshold).cast("double"))
    metrics = calculate_confusion_matrix_metrics(predictions, label_col, "pred_label")

    return {
        "threshold": threshold,
        "accuracy": round(metrics["accuracy"], 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1_score": round(metrics["f1_score"], 4),
        "specificity": round(metrics["specificity"], 4),
        "J_statistic": round(metrics["J_statistic"], 4)
    }

def find_best_threshold(predictions: DataFrame, label_col: str, probability_col: str, num_quantiles: int = 100):
    """Find the best threshold using Youden's J statistic on validation data."""
    predictions = predictions.withColumn("prob_positive", extract_prob_udf(F.col(probability_col)))
    quantiles = predictions.approxQuantile("prob_positive", [x / num_quantiles for x in range(num_quantiles + 1)], 0.01)

    best_threshold = 0.0
    best_j_stat = -1.0
    best_metrics = None

    for threshold in quantiles:
        metrics = evaluate_threshold(predictions, label_col, "prob_positive", threshold)
        if metrics["J_statistic"] > best_j_stat:
            best_j_stat = metrics["J_statistic"]
            best_threshold = threshold
            best_metrics = metrics

    print("\nBest Threshold Based on Youden's J Statistic:")
    for key, value in best_metrics.items():
        print(f"{key.replace('_', ' ').title()}: {value}")

    return best_threshold

def apply_threshold(predictions: DataFrame, probability_col: str, threshold: float):
    """Apply the best threshold to generate final predictions."""
    return predictions.withColumn("pred_label", (F.col("prob_positive") >= threshold).cast("double"))

# Transform validation data
val_data_pred = model.transform(valid_data)

# Find the best threshold using validation data
best_threshold = find_best_threshold(val_data_pred, "label", "probability")

# Apply the best threshold to validation data
val_data_pred = apply_threshold(val_data_pred, "probability", best_threshold)

# Save predictions to Parquet
val_data_pred.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/validation_predictions.parquet")

# Load saved validation predictions
val_data_pred = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/validation_predictions.parquet")

# Compute metrics for validation data
val_metrics = calculate_confusion_matrix_metrics(val_data_pred, "label", "pred_label")

# Print validation results
print(f"\nValidation Results:")
print(f"Accuracy: {val_metrics['accuracy']}")
print(f"Precision: {val_metrics['precision']}")
print(f"Recall (Sensitivity): {val_metrics['recall']}")
print(f"Specificity: {val_metrics['specificity']}")
print(f"F1 Score: {val_metrics['f1_score']}")
print(f"Best Threshold: {best_threshold}")

# Transform test data
test_data_pred = model.transform(test_data)

# Apply the best threshold to test data
test_data_pred = apply_threshold(test_data_pred, "probability", best_threshold)

# Save predictions to Parquet
test_data_pred.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/test_predictions.parquet")

# Load saved test predictions
test_data_pred = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/test_predictions.parquet")

# Compute metrics for test data
test_metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "pred_label")

# Print test results
print(f"\nTest Results:")
print(f"Accuracy: {test_metrics['accuracy']}")
print(f"Precision: {test_metrics['precision']}")
print(f"Recall (Sensitivity): {test_metrics['recall']}")
print(f"Specificity: {test_metrics['specificity']}")
print(f"F1 Score: {test_metrics['f1_score']}")

# Placeholder for AUC calculation
test_auc = "Test AUC calculation function needed"

print(f"Best Test AUC: {test_auc}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o785.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 13 in stage 339.0 failed 1 times, most recent failure: Lost task 13.0 in stage 339.0 (TID 3482, localhost, executor driver): java.io.EOFException: Cannot seek after EOF
	at org.apache.hadoop.fs.ChecksumFileSystem$FSDataBoundedInputStream.seek(ChecksumFileSystem.java:328)
	at org.apache.parquet.hadoop.util.H1SeekableInputStream.seek(H1SeekableInputStream.java:46)
	at org.apache.parquet.hadoop.ParquetFileReader.readFooter(ParquetFileReader.java:519)
	at org.apache.parquet.hadoop.ParquetFileReader.<init>(ParquetFileReader.java:689)
	at org.apache.parquet.hadoop.ParquetFileReader.open(ParquetFileReader.java:595)
	at org.apache.parquet.hadoop.ParquetRecordReader.initializeInternalReader(ParquetRecordReader.java:162)
	at org.apache.parquet.hadoop.ParquetRecordReader.initialize(ParquetRecordReader.java:140)
	at org.apache.spark.sql.execution.datasources.parquet.